In [5]:
import os
import re
import glob
import shutil
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from scipy.cluster import hierarchy
from scipy import cluster
from matplotlib.colors import LinearSegmentedColormap
from sklearn.preprocessing import MinMaxScaler
from matplotlib.colors import LogNorm
%matplotlib inline

In [ ]:
# method,threshold,normalization
def heatmap_plot(data,subtitle,filepath,is_save=False):
# # MinMaxScaler
#     minmax_scaler = MinMaxScaler()

# # DataFrame Min-Max normalization
#     minmax_normalized_data = minmax_scaler.fit_transform(data)

# # DataFrame
#     minmax_normalized_df = pd.DataFrame(minmax_normalized_data, columns=data.columns,index=data.index)

    minmax_normalized_df = data.copy()

    # norm = Normalize(vmin=0, vmax=1)
    # minmax_normalized_df = norm(minmax_normalized_df)
    # minmax_normalized_df = pd.DataFrame(minmax_normalized_df, columns=data.columns, index=data.index)
    #print(minmax_normalized_df.max(axis=1).max())
    max_median = minmax_normalized_df.max(axis=1).median()
    min_median = minmax_normalized_df.min(axis=1).median()
    min_min = minmax_normalized_df.min(axis=1).min()
    max_max = minmax_normalized_df.max(axis=1).max()
    cliped_df = minmax_normalized_df.clip(lower=min_median, upper=max_median)
    print(cliped_df.max(axis=1).max())
    cliped_df = (cliped_df-min_median)/(max_median-min_median)
    minmax_normalized_df = cliped_df
    index = np.linspace(0, 1, 8)
    colors = [(0,'#3333b2'),
              (index[1],'#3333b2'),
              (index[2],'#337fff'),
              (index[3],'#00b2b2'),
              (index[4],'#cccc33'),
              (index[5],'#ffb233'),
              (index[6],'#ffff19'),
              (1,'#ffff19')]
    #print(colors)
    cmap = LinearSegmentedColormap.from_list('custom_colormap',colors)

    #print(minmax_normalized_df.iloc[0,:]) 
    fig, ax = plt.subplots(figsize=(10, 3))
    #cliped_df = minmax_normalized_df.clip(upper=max_median)
    cax = ax.imshow(minmax_normalized_df, aspect='auto',cmap=cmap)
    cbar = fig.colorbar(cax, ax=ax)
    cbar.set_label('Value')
    # x y
    ax.set_xticks(np.arange(len(minmax_normalized_df.columns)))
    ax.set_yticks(np.arange(len(minmax_normalized_df.index)))
    ax.set_xticklabels(minmax_normalized_df.columns, rotation=90)
    #ax.set_yticklabels(minmax_normalized_df.index)
    ax.set_yticklabels([])

# (pdf)
    plt.title(subtitle)
    if is_save:
        plt.savefig(filepath + '.pdf',
                    format='pdf',
                    bbox_inches='tight', 
                    dpi=300,
                    transparent=True) 
    plt.show()
    plt.close()

    return minmax_normalized_df


In [7]:
project_path = '${PROJECT_ROOT}'
test = 'test1'

In [ ]:
#test1 params
cellType = 'CamkII_TST'#VIP:1~4;CamkII_TST:1~10;SST:1~3;PV:1~7;Hsyn:1~5
output_path = os.path.join(project_path,'summary',test,cellType)

for s in range(1,11):
    if test =='test2':
        print('test2 mode, skip this loop')
        break
    subject_dir = os.path.join(project_path,'summary',test,cellType,str(s))
    if not os.path.exists(subject_dir):
        print(f'Not find {cellType}{str(s)}')
        continue
    signal_dir = os.path.join(subject_dir,'signal_save')
    fig_dir =os.path.join(subject_dir,'event_4s_Heatmap')
    #if os.path.exists(fig_dir):shutil.rmtree(fig_dir)  !!!WARNING:line to delete current exising folder
    if not os.path.exists(fig_dir):os.makedirs(fig_dir)

    ###SI Analysis###

    #preprocess data 
    SIpattern = 'S_to_I_zscore_event_*.csv'
    matching_files = glob.glob(os.path.join(signal_dir, SIpattern))
    SI_avg_event = []
    for file in matching_files:
        df = pd.read_csv(file)
        df = df.drop(columns=['Unnamed: 0','response','upper','lower'])
        SI_avg_event.append(df)
    SI_avg_event = pd.concat(SI_avg_event, axis=0, keys=range(len(SI_avg_event)))
    # Get the values of 'upper' and 'lower' columns for the first row

    #find average event data
    SI_avg_event = SI_avg_event.groupby(level=1).mean()
    SI_avg_event.to_csv(os.path.join(signal_dir,'S_to_I_event_average_zscore.csv'))

    #remove baseline activity delta_f/f_b
    SI_baseline = SI_avg_event.iloc[:, 5:20]
    SI_noise = SI_baseline.mean(axis=1)
    SI_avg_event = SI_avg_event.subtract(SI_noise, axis=0)
    SI_avg_event = SI_avg_event.divide(SI_noise, axis=0)

    #sort by easilist firing neruons
    max_index = SI_avg_event.apply(lambda row: row.idxmax(), axis=1)
    max_index = pd.to_numeric(max_index)
    print(max_index.sort_values())
    sort_index = max_index.sort_values().index
    SI_avg_event = SI_avg_event.loc[sort_index]
    SI_avg_event.to_csv((os.path.join(signal_dir,'S_to_I_event_average_zscore_denoised.csv')))

    #Thersholding and 0-1 normalized is in the heatmap method
    normalized_df = heatmap_plot(SI_avg_event,f'{cellType}{str(s)}S to I event_4s_Heatmap',os.path.join(fig_dir,'S_to_I_average_zscore_heatmap'),True)
    # normalized_df.to_csv((os.path.join(fig_dir,'S_to_I_event_average_zscore_denoised_normalized.csv')))
    
    ###IS Analysis###
    ISpattern = 'I_to_S_zscore_event_*.csv'
    matching_files = glob.glob(os.path.join(signal_dir, ISpattern))
    IS_avg_event = []
    for file in matching_files:
        df = pd.read_csv(file)
        df = df.drop(columns=['Unnamed: 0','response','upper','lower'])
        IS_avg_event.append(df)
    IS_avg_event = pd.concat(IS_avg_event, axis=0, keys=range(len(IS_avg_event)))
    # Get the values of 'upper' and 'lower' columns for the first row
    #delta f / f
    IS_avg_event = IS_avg_event.groupby(level=1).mean()
    # IS_avg_event.to_csv((os.path.join(signal_dir,'I_to_S_event_average_zscore.csv')))
    #remove baseline activity delta_f/f_b
    IS_baseline = IS_avg_event.iloc[:, 5:20]
    IS_noise = IS_baseline.mean(axis=1)
    IS_avg_event = IS_avg_event.subtract(IS_noise, axis=0)
    IS_avg_event = IS_avg_event.divide(IS_noise, axis=0)

    #sort by easilist firing neruons
    max_index = IS_avg_event.apply(lambda row: row.idxmax(), axis=1)
    max_index = pd.to_numeric(max_index)
    print(max_index.sort_values())
    sort_index = max_index.sort_values().index
    IS_avg_event = IS_avg_event.loc[sort_index]
    IS_avg_event.to_csv((os.path.join(signal_dir,'I_to_S_event_average_zscore_denoised.csv')))

    #Thersholding and 0-1 normalized is in the heatmap method
    normalized_df = heatmap_plot(IS_avg_event,f'{cellType}{str(s)} I to S event_4s_Heatmap',os.path.join(fig_dir,'I_to_S_average_zscore_heatmap'),True)
    # normalized_df.to_csv((os.path.join(fig_dir,'I_to_S_event_average_zscore_denoised_normalized.csv')))


In [ ]:
#test2 params
condition = 'pre'#pre,post,rescue
experiment = 'TST'
mice_IDs = ["LHQ50","LHQ30","LH5799","LH5798","LH0166","LH0167","LH5779",
            "CSDS0087","CSDS0103","CSDS0126","CSDS0179","CSDS0370",
            "CSDSQ29", "CSDSQ27", "CSDSQ26", "CSDS5797", "CSDS5776"
]
mice_IDs = ['CSDS5776']
depressed = ["CSDS0126", "CSDS0087","CSDS0179","LH0167","CSDS5776","CSDS5797","CSDSQ26","LH5798","LHQ30","LHQ50"]
resist = ["CSDS0370","CSDS0103","LH0166","CSDSQ27","CSDSQ29","LH5799"] 

for mice in mice_IDs:
    if test =='test1':
        print('test1 mode, skip this loop')
        break
    subject_dir = os.path.join(project_path,'summary',test,condition,mice)
    if not os.path.exists(subject_dir):
        print(f'Not find {mice}{condition}')
        continue
    signal_dir = os.path.join(subject_dir,experiment,'signal_save')
    if not os.path.exists(signal_dir):
        print(f'Not find signal_save folder in {mice}{condition}')
        continue
    fig_dir =os.path.join(subject_dir,experiment,'event_4s_Heatmap')
    #if os.path.exists(fig_dir):shutil.rmtree(fig_dir)  !!!WARNING:line to delete current exising folder
    if not os.path.exists(fig_dir):os.makedirs(fig_dir)

    ###SI Analysis###
    #calculate average event activity
    SIpattern = 'S_to_I_zscore_event_*.csv'
    matching_files = glob.glob(os.path.join(signal_dir, SIpattern))
    SI_avg_event = []
    for file in matching_files:
        df = pd.read_csv(file)
        df = df.drop(columns=['Unnamed: 0','response','upper','lower'])
        SI_avg_event.append(df)
    SI_avg_event = pd.concat(SI_avg_event, axis=0, keys=range(len(SI_avg_event)))
    # Get the values of 'upper' and 'lower' columns for the first row
    SI_avg_event = SI_avg_event.groupby(level=1).mean()
    #SI_avg_event.to_csv(os.path.join(signal_dir,'S_to_I_event_average_zscore.csv'))

    #delta f / f
    #remove baseline activity delta_f/f_b
    SI_baseline = SI_avg_event.iloc[:, 5:20]
    SI_noise = SI_baseline.mean(axis=1)
    SI_avg_event = SI_avg_event.subtract(SI_noise, axis=0)
    SI_avg_event = SI_avg_event.divide(SI_noise, axis=0)

    #sort by easilist firing neruons
    max_index = SI_avg_event.apply(lambda row: row.idxmax(), axis=1)
    max_index = pd.to_numeric(max_index)
    print(max_index.sort_values())
    sort_index = max_index.sort_values().index
    SI_avg_event = SI_avg_event.loc[sort_index]
    # SI_avg_event.to_csv((os.path.join(signal_dir,'S_to_I_event_average_zscore_denoised.csv')))

    # #Thersholding and 0-1 normalized is in the heatmap method
    normalized_df = heatmap_plot(SI_avg_event,f'{mice}{condition}S to I event_4s_Heatmap',os.path.join(fig_dir,'S_to_I_average_zscore_heatmap'),True)
    # normalized_df.to_csv((os.path.join(fig_dir,'S_to_I_event_average_zscore_denoised_normalized.csv')))
    
    ###IS Analysis###
    ISpattern = 'I_to_S_zscore_event_*.csv'
    matching_files = glob.glob(os.path.join(signal_dir, ISpattern))
    IS_avg_event = []
    for file in matching_files:
        df = pd.read_csv(file)
        df = df.drop(columns=['Unnamed: 0','response','upper','lower'])
        IS_avg_event.append(df)
    IS_avg_event = pd.concat(IS_avg_event, axis=0, keys=range(len(IS_avg_event)))
    # Get the values of 'upper' and 'lower' columns for the first row
    #delta f / f
    IS_avg_event = IS_avg_event.groupby(level=1).mean()
    #IS_avg_event.to_csv((os.path.join(signal_dir,'I_to_S_event_average_zscore.csv')))
    #remove baseline activity delta_f/f_b
    IS_baseline = IS_avg_event.iloc[:, 5:20]
    IS_noise = IS_baseline.mean(axis=1)
    IS_avg_event = IS_avg_event.subtract(IS_noise, axis=0)
    IS_avg_event = IS_avg_event.divide(IS_noise, axis=0)

    #sort by easilist firing neruons
    max_index = IS_avg_event.apply(lambda row: row.idxmax(), axis=1)
    max_index = pd.to_numeric(max_index)
    print(max_index.sort_values())
    sort_index = max_index.sort_values().index
    IS_avg_event = IS_avg_event.loc[sort_index]
    # IS_avg_event.to_csv((os.path.join(signal_dir,'I_to_S_event_average_zscore_denoised.csv')))

    #Thersholding and 0-1 normalized is in the heatmap method
    normalized_df = heatmap_plot(IS_avg_event,f'{mice}{condition} I to S event_4s_Heatmap',os.path.join(fig_dir,'I_to_S_average_zscore_heatmap'),True)
    # normalized_df.to_csv((os.path.join(fig_dir,'I_to_S_event_average_zscore_denoised_normalized.csv')))
